# Notebook 3 — Why Bihar?
**YTS+ DSEB 2026 · Project B · Indian Railways**

---

## The question we are building toward

In Notebook 2 we discovered that India's most structurally critical railway stations are clustered in Bihar — not in Mumbai, Delhi, or Chennai.

Today we ask: **why Bihar?** And then: **why are the answer stations 160 years old?**

We will measure how far trains must travel compared to the straight-line distance. When those two numbers are very different, there is a physical barrier forcing a detour.

---
## Setup

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import time
import os
from math import radians, sin, cos, sqrt, atan2

os.makedirs('images', exist_ok=True)
DATA = 'data/'

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

print('Ready.')

In [ ]:
# Load stations and track network
stations = pd.read_csv(DATA + 'stations_clean.csv')
track_edges = pd.read_csv(DATA + 'track_edges.csv')

coords = stations.set_index('code')[['lat', 'lon']].to_dict('index')
name_lookup = stations.set_index('code')['name'].to_dict()
state_lookup = stations.set_index('code')['state'].to_dict()

print(f'Stations: {len(stations):,}')
print(f'Track edges: {len(track_edges):,}')

---
## Part 1 — Building a km-weighted network

In Notebooks 1 and 2 we used **number of trains** as the edge weight.

Now we want to measure **travel distance** — how many kilometres does a train travel to get from A to B?

For that we need a network where each edge weight is the **physical distance in kilometres** between adjacent stations.

In [ ]:
# Build km-weighted track graph
# Edge weight = haversine distance between the two stations

t0 = time.time()
G_km = nx.Graph()

for _, row in track_edges.iterrows():
    a, b = row['station_a'], row['station_b']
    if a in coords and b in coords:
        km = haversine(coords[a]['lat'], coords[a]['lon'],
                       coords[b]['lat'], coords[b]['lon'])
        G_km.add_edge(a, b, weight=km)

print(f'Km-weighted network built in {time.time()-t0:.2f}s')
print(f'  Nodes: {G_km.number_of_nodes():,}')
print(f'  Edges: {G_km.number_of_edges():,}')
print()
print('Now for any two stations, we can find:')
print('  Geodesic distance  = straight-line distance (haversine)')
print('  Train distance     = shortest path through the track network')

---
## Part 2 — Dijkstra from Patna: the Ganges question

**Patna** is the capital of Bihar. It sits on the south bank of the Ganges.

**Sonpur** is a small town directly across the river — about 11 km away as the crow flies.

How far do you have to travel by train to get from Patna to Sonpur?

We will use **Dijkstra's algorithm** — the standard algorithm for finding the shortest path in a weighted graph.

**Before you run: predict.** If Patna to Sonpur is about 11 km by straight line, how far do you think the shortest train route is? Why might it be longer?

*Your prediction:*

In [ ]:
# Dijkstra from Patna Junction (PNBE) to all reachable stations
SOURCE = 'PNBE'  # Patna Junction

if SOURCE not in G_km:
    print(f'{SOURCE} not in graph')
else:
    t0 = time.time()
    train_dists = nx.single_source_dijkstra_path_length(G_km, SOURCE, weight='weight')
    elapsed = time.time() - t0

    lat_s = coords[SOURCE]['lat']
    lon_s = coords[SOURCE]['lon']

    rows = []
    for code, train_km in train_dists.items():
        if code == SOURCE or code not in coords:
            continue
        lat_t, lon_t = coords[code]['lat'], coords[code]['lon']
        geo_km = haversine(lat_s, lon_s, lat_t, lon_t)
        if geo_km >= 20 and train_km > 0:
            rows.append({
                'code': code,
                'name': name_lookup.get(code, code),
                'state': state_lookup.get(code, ''),
                'geodesic_km': round(geo_km, 1),
                'train_km': round(train_km, 1),
                'ratio': round(train_km / geo_km, 2),
            })

    patna_df = pd.DataFrame(rows).sort_values('ratio', ascending=False)

    print(f'Dijkstra from Patna completed in {elapsed:.2f}s')
    print(f'Reachable stations (geodesic >= 20km): {len(patna_df):,}')
    print()
    print('Top 20 most indirect routes from Patna:')
    print('(stations that are close by straight line but far by train)')
    print()
    print(f'{"Code":<8} {"Name":<30} {"Straight-line":>13} {"By train":>9} {"Ratio":>7}')
    print('-' * 71)
    for _, row in patna_df.head(20).iterrows():
        print(f"{row['code']:<8} {str(row['name']):<30} {row['geodesic_km']:>10.1f} km {row['train_km']:>9.1f} km {row['ratio']:>7.1f}x")

**Write:** Look at the top few rows.
- The ratio tells you: the train travels X times further than the straight-line distance. What ratio does Patna → Sonpur show?
- Look at where these stations are (Bihar, UP). What do you think is causing these extreme detours?
- Why would there be a very long train detour between two towns that are only 20km apart?

*Your answer:*

In [ ]:
# Scatter: geodesic distance vs ratio, for all stations reachable from Patna
fig, ax = plt.subplots(figsize=(10, 5))

ax.scatter(patna_df['geodesic_km'], patna_df['ratio'],
           s=5, alpha=0.3, color='steelblue')

# Highlight extreme cases
extremes = patna_df[patna_df['ratio'] > 4]
ax.scatter(extremes['geodesic_km'], extremes['ratio'],
           s=20, alpha=0.7, color='tomato', zorder=4)

# Label a few
for _, row in patna_df.head(5).iterrows():
    ax.annotate(str(row['name'])[:16],
                (row['geodesic_km'], row['ratio']),
                fontsize=7, color='tomato',
                xytext=(4, 2), textcoords='offset points')

ax.axhline(1, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
ax.text(5, 1.05, 'Ratio = 1 (train = straight line)', fontsize=8)
ax.set_xlabel('Geodesic (straight-line) distance from Patna (km)')
ax.set_ylabel('Ratio: train km / straight-line km')
ax.set_title('From Patna: nearby stations that require huge train detours')
plt.tight_layout()
plt.savefig('images/patna_ratio_scatter.png', dpi=150)
plt.show()

print(f'Stations with ratio > 3: {(patna_df["ratio"] > 3).sum()}')
print(f'Stations with ratio > 5: {(patna_df["ratio"] > 5).sum()}')

---
## Part 3 — The global picture: where are the detours?

We pre-computed this comparison for all major station pairs across India. Let us load it and map it.

In [ ]:
# Load the global geodesic vs train data
# Columns: from_code, from_name, to_code, to_name, state, geodesic_km, train_km, ratio, detour_km
geo = pd.read_csv(DATA + 'geodesic_vs_train.csv')

print(f'Total high-ratio station pairs: {len(geo):,}')
print(f'(pairs where train is 2.5x+ longer than straight-line, geodesic 20-300km)')
print()
print('Distribution of ratios:')
print(geo['ratio'].describe().round(2))
print()
print('Top 20 highest-ratio pairs in India:')
print()
print(f'{"From":<25} {"To":<25} {"Straight-line":>13} {"By train":>9} {"Ratio":>7}')
print('-' * 81)
for _, row in geo.head(20).iterrows():
    print(f"{str(row['from_name']):<25} {str(row['to_name']):<25} {row['geodesic_km']:>10.1f} km {row['train_km']:>9.1f} km {row['ratio']:>7.1f}x")

**Write:** Look at the top 20. Which state appears most often? What region does the data suggest has the most geographic barriers to train travel?

*Your observation:*

In [ ]:
# Which states have the most high-ratio pairs?
state_counts = geo[geo['state'] != ''].groupby('state').size().sort_values(ascending=False)

print('High-ratio pairs by state (top 15):')
print(state_counts.head(15).to_string())

print()
print('Mean ratio by state (top 10):')
state_ratio = geo[geo['state'] != ''].groupby('state')['ratio'].mean().sort_values(ascending=False)
print(state_ratio.head(10).round(2).to_string())

In [ ]:
# Map the high-ratio pairs
# Use the FROM station coordinates, colored by ratio

# Only keep 'from' stations we have coordinates for
geo['from_lat'] = geo['from_code'].map(lambda c: coords[c]['lat'] if c in coords else None)
geo['from_lon'] = geo['from_code'].map(lambda c: coords[c]['lon'] if c in coords else None)
geo_plot = geo.dropna(subset=['from_lat', 'from_lon'])

# Clip ratio for color scale (extreme outliers compress the palette)
ratio_clipped = geo_plot['ratio'].clip(upper=12)

fig, ax = plt.subplots(figsize=(9, 11))

# All stations as background
all_codes = [c for c in G_km.nodes() if c in coords]
ax.scatter([coords[c]['lon'] for c in all_codes],
           [coords[c]['lat'] for c in all_codes],
           s=0.5, color='lightgrey', alpha=0.4, zorder=1)

sc = ax.scatter(geo_plot['from_lon'], geo_plot['from_lat'],
                c=ratio_clipped, cmap='YlOrRd',
                s=3, alpha=0.6, zorder=2, linewidths=0)
plt.colorbar(sc, ax=ax, label='Train/geodesic ratio (capped at 12)')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Geographic barriers to train travel — high-ratio station pairs')
ax.set_xlim(68, 97)
ax.set_ylim(8, 37)
plt.tight_layout()
plt.savefig('images/high_ratio_map.png', dpi=150)
plt.show()

**Write:** Look at the map. You should see two main clusters of high-ratio pairs.

1. **Bihar / eastern UP** — a cluster along a band running east-west. What geographic feature runs through this region?
2. **Gujarat (western India)** — another cluster around a single city. What might cause high ratios here? (Hint: think about how a city could act as a railway hub that all trains must pass through.)
3. **What is notably absent?** Look at the Deccan Plateau (Karnataka, Telangana, Maharashtra interior). Are there high-ratio pairs there? Why not?

*Your answer:*

---
## Part 4 — Two mechanisms, one map

The data shows two very different reasons why train distance can far exceed straight-line distance:

**Mechanism A — River barrier (Bihar, the Ganges)**

The Ganges river is extremely wide. There are very few rail bridges. To cross from the south bank to the north bank, a train must travel to one of the few bridge crossing points — even if that means going 150km out of the way.

**Mechanism B — Hub-spoke design (Gujarat, Ahmedabad)**

Ahmedabad was built as the regional hub of the railway network. Every line in Gujarat radiates outward from Ahmedabad. So to travel from one suburb to another suburb of Ahmedabad, a train must go: suburb → Ahmedabad → other suburb. A journey of 20km straight-line can become 300km by train.

**Which mechanism creates structural importance in betweenness?**

In [ ]:
# Compare Bihar vs Gujarat in the geodesic data
bihar = geo[geo['state'] == 'Bihar']
gujarat = geo[geo['state'] == 'Gujarat']

print('Bihar (river barrier mechanism):')
print(f'  High-ratio pairs: {len(bihar):,}')
print(f'  Mean ratio: {bihar["ratio"].mean():.2f}')
print(f'  Max ratio:  {bihar["ratio"].max():.2f}')
print()
print('Gujarat (hub-spoke mechanism):')
print(f'  High-ratio pairs: {len(gujarat):,}')
print(f'  Mean ratio: {gujarat["ratio"].mean():.2f}')
print(f'  Max ratio:  {gujarat["ratio"].max():.2f}')
print()
print('Top 5 Bihar high-ratio pairs:')
print(bihar[['from_name','to_name','geodesic_km','train_km','ratio']].head(5).to_string(index=False))
print()
print('Top 5 Gujarat high-ratio pairs:')
print(gujarat[['from_name','to_name','geodesic_km','train_km','ratio']].head(5).to_string(index=False))

**Write:** The river mechanism and the hub-spoke mechanism both create high train-to-geodesic ratios. But only ONE of them creates high betweenness centrality for specific stations. Which one? Why?

Think about it this way: in the hub-spoke case, if Ahmedabad disappeared, could trains re-route? In the river case, if the Kiul bridge disappeared, could trains re-route?

*Your answer:*

---
## Part 5 — When were these stations built?

We know the Bihar cluster stations are structurally critical. Now the question is: are they old or new?

**Your task:** For each station in the table below, search Wikipedia for the founding year. Write the year in the "Founded" column.

Search: `"[Station Name] railway station Wikipedia"`

In [ ]:
# Show the top betweenness stations with info for Wikipedia lookup
bc = pd.read_csv(DATA + 'betweenness_track.csv')
bc = bc.sort_values('betweenness_track', ascending=False).reset_index(drop=True)
bc['name'] = bc['code'].map(name_lookup).fillna(bc['code'])
bc['state'] = bc['code'].map(state_lookup).fillna('')

print('Top 15 stations by betweenness centrality')
print('Your task: find the founding year on Wikipedia')
print()
print(f'{"Rank":<5} {"Code":<8} {"Name":<30} {"State":<20} {"Founded"}')
print('-' * 80)
for i, row in bc.head(15).iterrows():
    print(f"{i+1:<5} {row['code']:<8} {str(row['name']):<30} {str(row['state']):<20} ___________")

**Wikipedia research task — do this on your phone or laptop:**

For each station, look up: when was this station opened / when did this railway line open?

Fill in the table (use the print cell above as your template):

| Rank | Code | Name | State | Founded |
|------|------|------|-------|---------|
| 1 | MB | Moradabad | | |
| 2 | KIUL | Kiul Junction | Bihar | |
| 3 | LKR | Luckeesarai Junction | Bihar | |
| 4 | SCC | Sitapur Cantonment | UP | |
| 5 | AII | Ajmer Junction | Rajasthan | |
| 6 | RNC | Ranchi | Jharkhand | |
| 7 | MKB | Mankatha | Bihar | |
| 8 | BRYA | Barhiya | Bihar | |
| 9 | PNBE | Patna Junction | Bihar | |
| 10 | DLI | Old Delhi | Delhi | |

*Your filled table:*

---
## Part 6 — The colonial question

Most of the top betweenness stations were built between **1860 and 1880** — more than 160 years ago.

This is not a coincidence.

The British colonial government in India had a specific goal when building the railway: connect **Calcutta** (the colonial capital, in Bengal) to the rest of the subcontinent for military and commercial movement of goods. Every line had to cross the Ganges at some point. There were very few places where the river was narrow enough to bridge.

The Bihar cluster stations — Kiul Junction, Luckeesarai, Hathidah Junction — sit at those original crossing points.

**160 years later, every shortest path still runs through the same handful of junctions the colonial engineers built first.**

In [ ]:
# If students found founding years, they can enter them here
# We provide a template — students fill in the years from Wikipedia

# Example (students should update these from their research):
found_data = {
    'Moradabad':          None,   # fill in from Wikipedia
    'Kiul Junction':      None,   # fill in from Wikipedia
    'Luckeesarai Junction': None, # fill in from Wikipedia
    'Sitapur Cantonment': None,   # fill in from Wikipedia
    'Ajmer Junction':     None,   # fill in from Wikipedia
    'Patna Junction':     None,   # fill in from Wikipedia
    'Old Delhi':          None,   # fill in from Wikipedia
    'New Delhi':          None,   # fill in from Wikipedia — this is more recent!
}

# Replace None with the year you found
# e.g. 'Kiul Junction': 1862

known = {k: v for k, v in found_data.items() if v is not None}

if len(known) >= 3:
    fig, ax = plt.subplots(figsize=(10, 5))
    names = list(known.keys())
    years = list(known.values())
    colors = ['tomato' if y < 1900 else 'steelblue' for y in years]
    bars = ax.barh(names, years, color=colors)
    ax.axvline(1947, color='black', linewidth=1.5, linestyle='--', label='Independence 1947')
    ax.axvline(1900, color='grey', linewidth=1, linestyle=':', alpha=0.7, label='Year 1900')
    ax.set_xlabel('Year founded')
    ax.set_title('When were India\'s most critical railway stations built?')
    ax.legend()
    plt.tight_layout()
    plt.savefig('images/founding_years.png', dpi=150)
    plt.show()
else:
    print('Fill in at least 3 founding years above, then re-run this cell.')
    print('You will see a bar chart showing when these stations were built.')

**Write — the big question:**

In Session 1 you were asked: *which Indian railway station, if it shut down, would cause the most disruption across the country — and why is the answer 160 years old?*

Write a paragraph that answers both parts:
1. **Which station** (and what the data shows)
2. **Why 160 years old** (the geographic and historical explanation)

*Your answer (this is the core of your final presentation):*

---
## Your final presentation

Your team will present for **10–12 minutes**. Here is the recommended structure:

**Slide 1 — The question** (30 seconds)
> *We set out to find which Indian railway station, if it shut down, would cause the most disruption — and whether the answer would surprise you.*

**Slides 2–3 — The network** (2 minutes)
- What is the network? Nodes, edges, what each means
- How we built physical track from train schedules (the skip edge insight)
- Show `all_stations_map.png` and `edge_distance_dist.png`

**Slides 4–5 — Degree** (1.5 minutes)
- What degree measures
- The top 20 — familiar names, spread across India
- Show `degree_map.png`

**Slides 6–7 — Betweenness** (2 minutes)
- What betweenness measures (the paths argument)
- The top 20 — very different, concentrated in Bihar
- Show `betweenness_map_india.png` and `degree_vs_betweenness.png`

**Slides 8–9 — Why Bihar** (2 minutes)
- The Ganges river forces all east-west traffic through a few bridge points
- Show `patna_ratio_scatter.png` and `high_ratio_map.png`
- The Patna → Sonpur example

**Slide 10 — The removal test** (1 minute)
- Remove Kiul: X stations stranded
- Remove New Delhi: Y stations stranded
- Show `removal_impact.png`

**Slide 11 — The colonial finding** (1.5 minutes)
- These stations were built 1860–1880 by the British
- The reason: the colonial route from Calcutta to the interior
- Show `founding_years.png` (if completed)

**Slide 12 — What the network cannot see** (1 minute)
- Our betweenness is unweighted — all paths treated equally
- We don't have passenger volumes or freight data
- What would change if we had that data?

**Slide 13 — Your answer** (30 seconds)
> *The station is ____. It is 160 years old because ____.*

---
## You are done.

You have built a physical track network for all of Indian Railways, identified the most structurally critical stations, and explained why the answer is a colonial-era junction in Bihar.

**For your presentation:** Make sure you can explain *why* you chose each graph or number — not just what the output says, but why it answers the question.